# Notebook 7: Data Augmentation Ablation

With only ~10 images per class, augmentation is critical. But which
transforms actually help? Remove them one at a time and find out.

We also **visualize** what each augmentation does to the image,
so you build intuition for what the model sees during training.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt
from torchvision import transforms, datasets
from src.data import IMAGENET_MEAN, IMAGENET_STD

DEVICE = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Visualize Each Augmentation

See the SAME flower image under different augmentations. This is what
the model sees during training — each epoch, each image looks slightly different.

In [ ]:
# Load one raw image (no transforms)
raw_dataset = datasets.Flowers102(root='../data', split='train', download=False,
                                   transform=transforms.ToTensor())
raw_img, label = raw_dataset[42]

# Define individual augmentations to visualize
augmentations = {
    'Original': transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)]),
    'RandomResizedCrop': transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    'HorizontalFlip': transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                                           transforms.RandomHorizontalFlip(p=1.0)]),
    'ColorJitter': transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                                       transforms.ColorJitter(0.3, 0.3, 0.3, 0.1)]),
    'Rotation(15°)': transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                                          transforms.RandomRotation(15)]),
    'All Combined': transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.2, 0.2, 0.2),
        transforms.RandomRotation(15)
    ])
}

# Show 3 random samples of each augmentation
from PIL import Image
pil_img = transforms.ToPILImage()(raw_img)

fig, axes = plt.subplots(len(augmentations), 4, figsize=(12, 2.5*len(augmentations)))
for row, (name, aug) in enumerate(augmentations.items()):
    axes[row][0].set_ylabel(name, fontsize=9, rotation=0, labelpad=80, va='center')
    for col in range(4):
        augmented = aug(pil_img)
        if isinstance(augmented, torch.Tensor):
            axes[row][col].imshow(augmented.permute(1, 2, 0).clamp(0, 1))
        else:
            axes[row][col].imshow(augmented)
        axes[row][col].axis('off')

plt.suptitle('Same Image Under Different Augmentations (4 random samples each)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/augmentation_samples.png', dpi=150)
plt.show()

## 2. Ablation: Remove One Augmentation at a Time

Train the same model with different augmentation configs.
The drop in accuracy when removing an augmentation = its contribution.

In [ ]:
from torch.utils.data import DataLoader
from src.model import create_model
from src.train import train_model

def make_train_transform(use_crop=True, use_flip=True, use_jitter=True, use_rotate=True):
    """Build a transform pipeline, optionally disabling specific augmentations."""
    steps = []
    if use_crop:
        steps.append(transforms.RandomResizedCrop(224, scale=(0.6, 1.0)))
    else:
        steps.extend([transforms.Resize(256), transforms.CenterCrop(224)])

    if use_flip:
        steps.append(transforms.RandomHorizontalFlip())
    if use_jitter:
        steps.append(transforms.ColorJitter(0.2, 0.2, 0.2))
    if use_rotate:
        steps.append(transforms.RandomRotation(15))

    steps.extend([
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
    ])
    return transforms.Compose(steps)


eval_transform = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

configs = {
    'All augmentations':  dict(use_crop=True,  use_flip=True,  use_jitter=True,  use_rotate=True),
    'No crop':            dict(use_crop=False, use_flip=True,  use_jitter=True,  use_rotate=True),
    'No flip':            dict(use_crop=True,  use_flip=False, use_jitter=True,  use_rotate=True),
    'No color jitter':    dict(use_crop=True,  use_flip=True,  use_jitter=False, use_rotate=True),
    'No rotation':        dict(use_crop=True,  use_flip=True,  use_jitter=True,  use_rotate=False),
    'No augmentation':    dict(use_crop=False, use_flip=False, use_jitter=False, use_rotate=False),
}

results = {}
NUM_EPOCHS = 15

for name, cfg in configs.items():
    print(f"\n{'='*50}")
    print(f"Config: {name}")
    print(f"{'='*50}")

    train_tf = make_train_transform(**cfg)
    train_set = datasets.Flowers102(root='../data', split='train', transform=train_tf)
    val_set = datasets.Flowers102(root='../data', split='val', transform=eval_transform)

    train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=2)

    model = create_model(num_classes=102, mode='feature_extract').to(DEVICE)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(params, lr=1e-3)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

    history = train_model(
        model, train_loader, val_loader, DEVICE,
        optimizer=optimizer, scheduler=scheduler, num_epochs=NUM_EPOCHS,
        save_path=f'../models/aug_{name.replace(" ", "_")}.pth'
    )

    results[name] = {'best_val': max(history['val_acc']),
                     'final_train': history['train_acc'][-1],
                     'history': history}

In [ ]:
# ---- Summary plot ----
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

names = list(results.keys())
val_accs = [results[n]['best_val'] for n in names]
train_accs = [results[n]['final_train'] for n in names]
gaps = [t - v for t, v in zip(train_accs, val_accs)]

# Accuracy comparison
x = range(len(names))
axes[0].barh(x, val_accs, color='steelblue')
axes[0].set_yticks(x)
axes[0].set_yticklabels(names)
axes[0].set_xlabel('Best Validation Accuracy (%)')
axes[0].set_title('Effect of Removing Each Augmentation')
axes[0].grid(True, alpha=0.3, axis='x')

# Overfitting gap
colors = ['#2ecc71' if g < 15 else '#f39c12' if g < 30 else '#e74c3c' for g in gaps]
axes[1].barh(x, gaps, color=colors)
axes[1].set_yticks(x)
axes[1].set_yticklabels(names)
axes[1].set_xlabel('Train - Val Accuracy Gap (%)')
axes[1].set_title('Overfitting Gap (smaller = better generalization)')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('../results/augmentation_ablation.png', dpi=150)
plt.show()